# Point-in-Time Feature Datamart by `as_of_date`

Построение переиспользуемых признаков для кандидитогенераторов и реранкера в формате point-in-time feature store. Признаки сохраняются не как `train/val/test`, а как срезы на дату `as_of_date`.

In [1]:
from __future__ import annotations

import gc
import json
import shutil
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Iterable

import pandas as pd
import polars as pl
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pl.Config.set_tbl_cols(200)
pl.Config.set_tbl_rows(30)

polars.config.Config

## 1. Paths and config

In [2]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("Не найден project root с папкой data")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = DATA_DIR / "datamart"
AGG_DIR = DATAMART_DIR / "agg"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATAMART_DIR:", DATAMART_DIR)
print("AGG_DIR:", AGG_DIR)

PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
DATAMART_DIR: c:\Users\denis\Documents\Repo\ozon\data\datamart
AGG_DIR: c:\Users\denis\Documents\Repo\ozon\data\datamart\agg


In [ ]:
@dataclass(frozen=True)
class FeatureDatamartConfig:
    feature_store_version: str = "reranker_v1"

    # name is only an alias for logs; physical partition is as_of_date.
    feature_specs: tuple[tuple[str, date, date], ...] = (
        ("ranker_train", date(2025, 4, 18), date(2025, 6, 3)),
        ("val", date(2025, 4, 18), date(2025, 6, 17)),
        ("test", date(2025, 5, 2), date(2025, 7, 1)),
    )

    # Reusable, reasonably compact feature levels.
    # user_item is excluded because full user-item graph is usually too large.
    feature_levels: tuple[str, ...] = ("user", "item", "category", "user_category")

    # Rolling windows ending at as_of_date, inclusive.
    feature_windows: tuple[int, ...] = (7, 14, 30, 60)

    # Set True only when feature definitions changed or data was rebuilt.
    rebuild_features: bool = False


FCFG = FeatureDatamartConfig()
FEATURES_DIR = DATAMART_DIR / "features" / FCFG.feature_store_version
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("FEATURES_DIR:", FEATURES_DIR)
FCFG

FEATURES_DIR: c:\Users\denis\Documents\Repo\ozon\data\datamart\features\reranker_v1


FeatureDatamartConfig(feature_store_version='reranker_v1', feature_specs=(('ranker_train', datetime.date(2025, 4, 18), datetime.date(2025, 6, 3)), ('val', datetime.date(2025, 4, 18), datetime.date(2025, 6, 17)), ('test', datetime.date(2025, 5, 2), datetime.date(2025, 7, 1))), feature_levels=('user', 'item', 'category', 'user_category'), feature_windows=(7, 14, 30, 60), rebuild_features=False)

## 2. Common helpers

In [4]:
EVENT_METRIC_COLS = (
    "cnt_events",
    "cnt_page_view",
    "cnt_view_description",
    "cnt_review_view",
    "cnt_to_cart",
    "cnt_favorite",
    "cnt_remove",
    "cnt_unfavorite",
)

ORDER_METRIC_COLS = (
    "cnt_orders",
    "cnt_delivered_orders",
    "cnt_canceled_orders",
    "cnt_proccesed_orders",
)

LEVEL_KEYS = {
    "user": ["user_id"],
    "item": ["item_id"],
    "user_item": ["user_id", "item_id"],
    "user_category": ["user_id", "catalogid"],
    "category": ["catalogid"],
}


def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Iterable[str], dtype=pl.Float32) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    missing_exprs = [pl.lit(0).cast(dtype).alias(c) for c in cols if c not in existing]
    return lf.with_columns(missing_exprs) if missing_exprs else lf


def clean_output_path(path: Path) -> None:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
    path.parent.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)

## 3. Read datamart aggregates point-in-time

In [ ]:
def events_path(level: str) -> Path:
    return AGG_DIR / "events" / level


def orders_path(level: str) -> Path:
    return AGG_DIR / "orders" / level


def _date_filter(lf: pl.LazyFrame, date_col: str, start: date, end: date) -> pl.LazyFrame:
    return (
        lf.with_columns(pl.col(date_col).cast(pl.Date, strict=False).alias(date_col))
        .filter(
            (pl.col(date_col) >= pl.lit(start).cast(pl.Date))
            & (pl.col(date_col) <= pl.lit(end).cast(pl.Date))
        )
    )


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    return _date_filter(scan_parquet(events_path(level), hive_partitioning=True), "event_date", start, end)


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    return _date_filter(scan_parquet(orders_path(level), hive_partitioning=True), "created_date", start, end)


def item_catalog_lf() -> pl.LazyFrame:
    path = DATAMART_DIR / "mappings" / "item_catalog.parquet"
    if path.exists():
        return scan_parquet(path, hive_partitioning=False).select(
            pl.col("item_id").cast(pl.Int64),
            pl.col("catalogid").cast(pl.Int64),
        ).unique()

    # Fallback: infer from user_item aggregates if the mapping file was not built.
    for candidate in [events_path("user_item"), orders_path("user_item")]:
        if path_has_parquet(candidate):
            lf = scan_parquet(candidate, hive_partitioning=True)
            if "catalogid" in lazy_columns(lf):
                return lf.select(
                    pl.col("item_id").cast(pl.Int64),
                    pl.col("catalogid").cast(pl.Int64),
                ).drop_nulls().unique()

    raise FileNotFoundError(
        "Не найден item->catalogid mapping. Ожидался data/datamart/mappings/item_catalog.parquet "
        "или catalogid в agg/events/user_item | agg/orders/user_item."
    )


def source_lf_for_level(domain: str, level: str, start: date, end: date) -> pl.LazyFrame | None:
    if domain == "events":
        path_fn = events_path
        read_fn = read_events_lf
    elif domain == "orders":
        path_fn = orders_path
        read_fn = read_orders_lf
    else:
        raise ValueError(domain)

    if path_has_parquet(path_fn(level)):
        return read_fn(level, start, end)

    if level != "user_item" and path_has_parquet(path_fn("user_item")):
        return read_fn("user_item", start, end)

    return None


def prepare_source_keys_for_level(lf: pl.LazyFrame, level: str) -> pl.LazyFrame:
    cols = set(lazy_columns(lf))

    if "catalogid" in LEVEL_KEYS[level] and "catalogid" not in cols:
        if "item_id" not in cols:
            raise ValueError(f"Cannot build {level}: source has neither catalogid nor item_id")
        lf = lf.join(item_catalog_lf(), on="item_id", how="left")
        cols.add("catalogid")

    casts = []
    for key in LEVEL_KEYS[level]:
        if key not in cols and key not in lazy_columns(lf):
            raise ValueError(f"Cannot build {level}: missing key column {key}")
        casts.append(pl.col(key).cast(pl.Int64, strict=False).alias(key))
    return lf.with_columns(casts)

## 4. Aggregate feature definitions

In [6]:
def safe_ratio_expr(num_col: str, den_col: str, out_col: str) -> pl.Expr:
    num = pl.col(num_col).cast(pl.Float32).fill_null(0.0)
    den = pl.col(den_col).cast(pl.Float32).fill_null(0.0)
    return (
        pl.when(den > 0.0)
        .then(num / den)
        .otherwise(pl.lit(None))
        .cast(pl.Float32)
        .alias(out_col)
    )


def aggregate_events_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("events", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, EVENT_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in EVENT_METRIC_COLS
    ]
    agg_exprs.append(pl.max("event_date").alias("__last_event_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_events = f"{prefix}cnt_events_{window}d"
    cnt_page = f"{prefix}cnt_page_view_{window}d"
    cnt_desc = f"{prefix}cnt_view_description_{window}d"
    cnt_review = f"{prefix}cnt_review_view_{window}d"
    cnt_cart = f"{prefix}cnt_to_cart_{window}d"
    cnt_fav = f"{prefix}cnt_favorite_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_event_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_event_{window}d"),
        (pl.col(cnt_events).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_events_{window}d"),
        safe_ratio_expr(cnt_cart, cnt_page, f"{prefix}to_cart_per_page_view_{window}d"),
        safe_ratio_expr(cnt_fav, cnt_page, f"{prefix}favorite_per_page_view_{window}d"),
        safe_ratio_expr(cnt_desc, cnt_page, f"{prefix}description_per_page_view_{window}d"),
        safe_ratio_expr(cnt_review, cnt_page, f"{prefix}review_per_page_view_{window}d"),
    ).drop("__last_event_date")


def aggregate_orders_level(level: str, start: date, end: date, window: int) -> pl.LazyFrame | None:
    keys = LEVEL_KEYS[level]
    lf = source_lf_for_level("orders", level, start, end)
    if lf is None:
        return None

    lf = prepare_source_keys_for_level(lf, level)
    lf = ensure_numeric_cols(lf, ORDER_METRIC_COLS)

    prefix = f"f_num__{level}__"
    agg_exprs = [
        pl.sum(c).cast(pl.Float32).alias(f"{prefix}{c}_{window}d")
        for c in ORDER_METRIC_COLS
    ]
    agg_exprs.append(pl.max("created_date").alias("__last_order_date"))

    out = lf.group_by(keys).agg(agg_exprs)

    cnt_orders = f"{prefix}cnt_orders_{window}d"
    cnt_delivered = f"{prefix}cnt_delivered_orders_{window}d"
    cnt_cancel = f"{prefix}cnt_canceled_orders_{window}d"
    cnt_processed = f"{prefix}cnt_proccesed_orders_{window}d"

    return out.with_columns(
        (pl.lit(end).cast(pl.Date) - pl.col("__last_order_date").cast(pl.Date))
        .dt.total_days()
        .cast(pl.Float32)
        .alias(f"{prefix}days_since_last_order_{window}d"),
        (pl.col(cnt_orders).fill_null(0.0) > 0).cast(pl.Int8).alias(f"{prefix}has_orders_{window}d"),
        safe_ratio_expr(cnt_delivered, cnt_orders, f"{prefix}delivered_share_{window}d"),
        safe_ratio_expr(cnt_cancel, cnt_orders, f"{prefix}cancel_share_{window}d"),
        safe_ratio_expr(cnt_processed, cnt_orders, f"{prefix}processed_share_{window}d"),
    ).drop("__last_order_date")

## 5. Build feature partitions by date

In [7]:
def feature_date_dir(as_of_date: date) -> Path:
    return FEATURES_DIR / f"as_of_date={as_of_date.isoformat()}"


def build_level_features_asof(alias: str, context_start: date, as_of_date: date, level: str) -> Path | None:
    if level not in LEVEL_KEYS:
        raise ValueError(f"Unknown level: {level}. Available: {sorted(LEVEL_KEYS)}")

    out_dir = feature_date_dir(as_of_date)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{level}.parquet"

    if out_path.exists() and not FCFG.rebuild_features:
        print(f"{alias}/{level}: exists -> {rel(out_path)}")
        return out_path

    clean_output_path(out_path)
    parts: list[pl.LazyFrame] = []

    for window in FCFG.feature_windows:
        win_start = max(context_start, as_of_date - timedelta(days=int(window) - 1))

        ev = aggregate_events_level(level, win_start, as_of_date, int(window))
        if ev is not None:
            parts.append(ev)

        orders = aggregate_orders_level(level, win_start, as_of_date, int(window))
        if orders is not None:
            parts.append(orders)

    if not parts:
        print(f"{alias}/{level}: no sources")
        return None

    keys = LEVEL_KEYS[level]
    base_keys = pl.concat([p.select(keys) for p in parts], how="vertical_relaxed").unique()

    features_lf = base_keys
    for part in parts:
        features_lf = features_lf.join(part, on=keys, how="left")

    df = collect_streaming(features_lf)
    if df.is_empty():
        print(f"{alias}/{level}: empty")
        return None

    df.write_parquet(out_path, compression="zstd")
    print(f"{alias}/{level}: rows={df.height:,}, cols={df.width:,} -> {rel(out_path)}")
    del df
    gc.collect()
    return out_path


def build_feature_store_asof(alias: str, context_start: date, as_of_date: date) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    for level in tqdm(FCFG.feature_levels, desc=f"as_of_date={as_of_date.isoformat()}"):
        path = build_level_features_asof(alias, context_start, as_of_date, level)
        if path is not None:
            paths[level] = path
    return paths


def write_manifest(results: dict[str, dict[str, str]]) -> Path:
    manifest_path = FEATURES_DIR / "manifest.json"
    payload = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "config": asdict(FCFG),
        "results": results,
    }
    manifest_path.write_text(json.dumps(payload, indent=2, default=str, ensure_ascii=False), encoding="utf-8")
    return manifest_path

In [8]:
results: dict[str, dict[str, str]] = {}
for alias, context_start, as_of_date in FCFG.feature_specs:
    paths = build_feature_store_asof(alias, context_start, as_of_date)
    results[alias] = {level: rel(path) for level, path in paths.items()}

manifest_path = write_manifest(results)
print("manifest:", rel(manifest_path))
results

as_of_date=2025-06-03:   0%|          | 0/4 [00:00<?, ?it/s]

ranker_train/user: rows=943,558, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-03\user.parquet
ranker_train/item: rows=4,311,566, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-03\item.parquet
ranker_train/category: rows=464, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-03\category.parquet
ranker_train/user_category: rows=31,675,400, cols=94 -> data\datamart\features\reranker_v1\as_of_date=2025-06-03\user_category.parquet


as_of_date=2025-06-17:   0%|          | 0/4 [00:00<?, ?it/s]

val/user: rows=954,741, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-17\user.parquet
val/item: rows=4,429,293, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-17\item.parquet
val/category: rows=465, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-06-17\category.parquet
val/user_category: rows=36,690,163, cols=94 -> data\datamart\features\reranker_v1\as_of_date=2025-06-17\user_category.parquet


as_of_date=2025-07-01:   0%|          | 0/4 [00:00<?, ?it/s]

test/user: rows=952,025, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-07-01\user.parquet
test/item: rows=4,377,370, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-07-01\item.parquet
test/category: rows=464, cols=93 -> data\datamart\features\reranker_v1\as_of_date=2025-07-01\category.parquet
test/user_category: rows=36,279,806, cols=94 -> data\datamart\features\reranker_v1\as_of_date=2025-07-01\user_category.parquet
manifest: data\datamart\features\reranker_v1\manifest.json


{'ranker_train': {'user': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-03\\user.parquet',
  'item': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-03\\item.parquet',
  'category': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-03\\category.parquet',
  'user_category': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-03\\user_category.parquet'},
 'val': {'user': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-17\\user.parquet',
  'item': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-17\\item.parquet',
  'category': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-17\\category.parquet',
  'user_category': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-06-17\\user_category.parquet'},
 'test': {'user': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-07-01\\user.parquet',
  'item': 'data\\datamart\\features\\reranker_v1\\as_of_date=2025-07-01\\item.parquet',
  'category': 'data\\datamart\\feat

## 6. Sanity checks

In [9]:
def feature_store_inventory() -> pd.DataFrame:
    rows = []
    for alias, _, as_of_date in FCFG.feature_specs:
        root = feature_date_dir(as_of_date)
        for level in FCFG.feature_levels:
            path = root / f"{level}.parquet"
            if not path.exists():
                rows.append({"alias": alias, "as_of_date": as_of_date, "level": level, "exists": False})
                continue
            lf = scan_parquet(path, hive_partitioning=False)
            cols = lazy_columns(lf)
            summary = collect_streaming(lf.select(pl.len().alias("rows")))
            rows.append({
                "alias": alias,
                "as_of_date": as_of_date,
                "level": level,
                "exists": True,
                "rows": int(summary["rows"][0]),
                "cols": len(cols),
                "path": rel(path),
            })
    return pd.DataFrame(rows)


inventory_df = feature_store_inventory()
inventory_df

,alias,as_of_date,level,exists,rows,cols,path
0,ranker_train,2025-06-03,user,True,943558,93,data\datamart\features\reranker_v1\as_of_date=...
1,ranker_train,2025-06-03,item,True,4311566,93,data\datamart\features\reranker_v1\as_of_date=...
2,ranker_train,2025-06-03,category,True,464,93,data\datamart\features\reranker_v1\as_of_date=...
3,ranker_train,2025-06-03,user_category,True,31675400,94,data\datamart\features\reranker_v1\as_of_date=...
4,val,2025-06-17,user,True,954741,93,data\datamart\features\reranker_v1\as_of_date=...
5,val,2025-06-17,item,True,4429293,93,data\datamart\features\reranker_v1\as_of_date=...
6,val,2025-06-17,category,True,465,93,data\datamart\features\reranker_v1\as_of_date=...
7,val,2025-06-17,user_category,True,36690163,94,data\datamart\features\reranker_v1\as_of_date=...
8,test,2025-07-01,user,True,952025,93,data\datamart\features\reranker_v1\as_of_date=...
9,test,2025-07-01,item,True,4377370,93,data\datamart\features\reranker_v1\as_of_date=...


## 7. Candidate-scoped features outside common datamart

Полный граф `user_item` намеренно не материализуется в общем feature store. Такие признаки строятся только для пар `user_id`–`item_id`, которые уже попали в candidate pool реранкера.